# 🎭 ChuckleNet: Process Local Files (No YouTube Download!)

**IMPORTANT**: Upload your local files to Google Drive FIRST:

```bash
# On your LOCAL machine, run:
rclone copy /Users/Subho/data/utterances/vtt_audio_local/ gdrive:chuckle_net/audio/ --progress
rclone copy /Users/Subho/data/chuckle_vtt_labels/ gdrive:chuckle_net/vtt/ --progress
```

Or drag-and-drop via browser at: drive.google.com

**What this does**:
- Process 620+ videos you already have
- Re-extract F0 at VTT utterance boundaries (not 1s fixed segments)
- Label by [laughter] in captions
- Train F0 model with multilingual evaluation

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3

import os, glob
import numpy as np
from tqdm import tqdm

BASE = '/content/drive/My Drive/chuckle_net'
AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

# Check files exist
audio_files = glob.glob(f'{AUDIO_DIR}/*.wav') + glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.mp3')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')

print(f'Audio files: {len(audio_files)}')
print(f'VTT files: {len(vtt_files)}')
print(f'Audio dir: {AUDIO_DIR}')
print(f'VTT dir: {VTT_DIR}')

In [ ]:
# 2. Load 87-video dataset (already verified, F1=0.9553)
import numpy as np

# Download from GitHub LFS
!wget -q -O /tmp/87_video_features.npz 'https://github.com/Das-rebel/ChuckleNet/raw/main/data/wavlm_training_data_expanded.npz' 2>/dev/null || echo 'Will use local path'

# Try alternative: load from Drive if uploaded
local_87 = '/content/drive/My Drive/chuckle_net/wavlm_training_data_expanded.npz'
if os.path.exists(local_87):
    d87 = np.load(local_87, allow_pickle=True)
    X87 = d87['features']
    y87 = d87['labels']
    print(f'87-video dataset: {X87.shape}, {100*y87.mean():.1f}% positive')
else:
    print('87-video dataset NOT found in Drive')
    print('Download from: https://github.com/Das-rebel/ChuckleNet/blob/main/data/wavlm_training_data_expanded.npz')

In [ ]:
# 3. Parse VTT and extract F0 at utterance boundaries
import librosa, re

def parse_vtt_cues(vtt_path):
    """Parse VTT, return list of (start_sec, end_sec, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    def to_sec(ts):
        ts = ts.replace('.', ':')
        p = ts.split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    
    cues = []
    lines = content.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if '-->' in line:
            start, end = line.split('-->')
            start, end = to_sec(start.strip()), to_sec(end.strip())
            text_lines = []
            i += 1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                text_lines.append(lines[i].strip())
                i += 1
            text = ' '.join(text_lines)
            has_laughter = '[laughter]' in text.lower()
            cues.append((start, end, text, has_laughter))
        else:
            i += 1
    return cues

def extract_f0_features(y, sr=22050, hop_length=512):
    """Extract 5-dim F0 features from audio segment."""
    try:
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=hop_length)
        f0 = np.nan_to_num(f0, nan=0.0)
        return [
            float(np.mean(f0)), float(np.std(f0)),
            float(np.max(f0)), float(np.min(f0)),
            float(np.mean(f0 > 0))
        ]
    except:
        return [0, 0, 0, 0, 0]

def process_video(audio_path, vtt_path):
    """Process one video: parse VTT + extract F0 at utterance boundaries."""
    cues = parse_vtt_cues(vtt_path)
    if not cues:
        return None, None, None
    
    try:
        y, sr = librosa.load(audio_path, sr=22050)
    except:
        return None, None, None
    
    features, labels, uid_list = [], [], []
    for i, (start, end, text, has_laughter) in enumerate(cues):
        if end <= start or end - start > 30:  # Skip invalid or too long
            continue
        y_seg = y[int(start*sr):int(end*sr)]
        if len(y_seg) < sr * 0.1:
            continue
        feat = extract_f0_features(y_seg, sr)
        vid = os.path.basename(audio_path).split('.')[0]
        features.append(feat)
        labels.append(1 if has_laughter else 0)
        uid_list.append(f'{vid}_{i}')
    
    return np.array(features), np.array(labels), uid_list

# Test
if audio_files and vtt_files:
    test_audio = audio_files[0]
    vid = os.path.basename(test_audio).split('.')[0]
    test_vtt = f'{VTT_DIR}/{vid}.en.vtt'
    if not os.path.exists(test_vtt):
        # Try without .en
        test_vtt = f'{VTT_DIR}/{vid}.vtt'
    if os.path.exists(test_vtt):
        feats, labs, uids = process_video(test_audio, test_vtt)
        print(f'Test: {len(feats)} segments, {labs.sum()} positive ({100*labs.mean():.1f}%)')
    else:
        print(f'VTT not found for {vid}')
        print(f'Looking in: {VTT_DIR}')
        print(f'VTT files sample: {[os.path.basename(v) for v in vtt_files[:3]]}')

In [ ]:
# 4. Batch Process All Videos
all_features, all_labels, all_vids, all_uids, all_langs = [], [], [], [], []

# Build audio→VTT mapping
for audio_path in tqdm(audio_files, desc='Finding matches'):
    vid = os.path.basename(audio_path).split('.')[0]
    # Try different VTT naming patterns
    vtt_path = None
    for pattern in [f'{vid}.en.vtt', f'{vid}.vtt', f'{vid}.hi.vtt', f'{vid}.zh.vtt']:
        candidate = f'{VTT_DIR}/{pattern}'
        if os.path.exists(candidate):
            vtt_path = candidate
            break
    if vtt_path is None:
        continue
    
    # Detect language from VTT filename
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    try:
        feats, labs, uids = process_video(audio_path, vtt_path)
        if feats is None or len(feats) == 0:
            continue
        for f, l, u in zip(feats, labs, uids):
            all_features.append(f)
            all_labels.append(l)
            all_vids.append(vid)
            all_uids.append(u)
            all_langs.append(lang)
    except Exception as e:
        continue

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)
uids = np.array(all_uids)
langs = np.array(all_langs)

print(f'\nTotal: {len(X)} segments from {len(set(vids))} videos')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% positive')

In [ ]:
# 5. Combine with 87-video dataset (if available)
if 'X87' in dir() and 'y87' in dir():
    print('Combining with 87-video dataset...')
    # The 87-video uses different feature format (23-dim prosody)
    # We only use the 5-dim F0 portion for comparison
    X87_f0 = X87[:, :5]  # First 5 features are F0
    
    # Align feature names
    print(f'87-video: {X87_f0.shape}, {100*y87.mean():.1f}% positive')
    print(f'New data: {X.shape}, {100*y.mean():.1f}% positive')
    
    # Combine
    X_combined = np.vstack([X87_f0, X])
    y_combined = np.hstack([y87, y])
    
    print(f'Combined: {X_combined.shape}, {100*y_combined.mean():.1f}% positive')
else:
    X_combined = X
    y_combined = y
    print('Using new data only (87-video not available)')

In [ ]:
# 6. Train + Evaluate (Video-level split)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

DATA = X_combined if 'X_combined' in dir() else X
LABELS = y_combined if 'y_combined' in dir() else y

# Split by VIDEO (not random)
unique_vids = list(set(vids))
np.random.seed(42)
np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids) * 0.2))
test_vids = set(unique_vids[:n_test])
train_vids = set(unique_vids[n_test:])

train_mask = np.isin(vids, list(train_vids))
test_mask = ~train_mask

X_train, X_test = DATA[train_mask], DATA[test_mask]
y_train, y_test = LABELS[train_mask], LABELS[test_mask]

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% positive)')
print(f'Test:  {len(X_test)} ({100*y_test.mean():.1f}% positive)')

# LR
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(f'\n=== Logistic Regression ===')
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

# MLP
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print(f'\n=== MLP ===')
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 7. Multilingual F1 Breakdown
print('=== F1 by Language ===')
for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() < 50:
        continue
    
    lang_test = np.isin(vids, list(test_vids)) & mask
    if lang_test.sum() < 10:
        continue
    
    y_pred_lang = lr.predict(DATA[lang_test])
    f1 = f1_score(LABELS[lang_test], y_pred_lang)
    prec = precision_score(LABELS[lang_test], y_pred_lang)
    rec = recall_score(LABELS[lang_test], y_pred_lang)
    print(f'{lang}: F1={f1:.4f} P={prec:.4f} R={rec:.4f} ({lang_test.sum()} test segs)')

In [ ]:
# 8. Save
out = {
    'features': X,
    'labels': y,
    'vids': vids,
    'uids': uids,
    'langs': langs
}
np.savez_compressed(f'{BASE}/processed_local_620.npz', **out)
print(f'Saved: {BASE}/processed_local_620.npz')

import pickle
with open(f'{BASE}/f0_model_local.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Model: {BASE}/f0_model_local.pkl')

print('\n✅ DONE!')